# Arricchimento Titoli via Cellar SPARQL

Questo notebook recupera i titoli degli atti normativi del grafo focale tramite l'**API SPARQL ufficiale di EUR-Lex (Cellar)**.

## Perché questo step è necessario

Il dataset EurLex non include i titoli degli atti. I titoli sono fondamentali per la classificazione nei layer Lamfalussy perché contengono informazioni strutturali esplicite non presenti nei metadati:

| Titolo | Layer |
|---|---|
| `Regulation of the European Parliament and of the Council` | **L1** |
| `Commission Delegated Regulation` | **L2** |
| `Commission Implementing Regulation` | **L2** |
| `Guidelines of the European Banking Authority` | **L3** |
| `Judgment of the Court of Justice` | **L4** |

## Parametri
- **API**: endpoint SPARQL Cellar, nessun WAF
- **Delay**: 0.5s tra richieste
- **Checkpoint**: ogni 50 nodi
- **Input**: `data/output/golden_power/gephi_nodes_focal.csv`
- **Output**: `data/output/golden_power/gephi_nodes_focal_titled.csv`

In [ ]:
import pandas as pd
import requests
import time
import os
import sys

sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'gephi_nodes_focal.csv')
output_file     = os.path.join(output_path, 'gephi_nodes_focal_titled.csv')
checkpoint_file = os.path.join(output_path, 'titles_checkpoint.csv')

SPARQL_ENDPOINT  = 'https://publications.europa.eu/webapi/rdf/sparql'
DELAY_SECONDS    = 0.5
CHECKPOINT_EVERY = 50
TIMEOUT          = 15

print(f"Input:      {input_file}")
print(f"Output:     {output_file}")
print(f"Checkpoint: {checkpoint_file}")

## 1. Caricamento Nodi e Gestione Checkpoint

In [ ]:
nodes = pd.read_csv(input_file)
print(f"Nodi totali nel grafo focale: {len(nodes)}")

if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi gia processati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=['Id', 'Label', 'title', 'title_status'])
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[~nodes['Id'].isin(already_done)].copy()
print(f"Da processare ora: {len(nodes_todo)}")

## 2. Funzione SPARQL

In [ ]:
def get_title_from_cellar(celex, timeout=TIMEOUT):
    """
    Recupera il titolo inglese di un atto dall'API SPARQL di Cellar.
    Restituisce (title, status).
    """
    if pd.isna(celex) or celex == '':
        return None, 'not_found'

    query = f"""
    PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
    SELECT ?title WHERE {{
      ?work cdm:resource_legal_id_celex "{celex}"^^<http://www.w3.org/2001/XMLSchema#string> .
      ?expr cdm:expression_belongs_to_work ?work .
      ?expr cdm:expression_title ?title .
      FILTER(lang(?title) = "en")
    }}
    LIMIT 1
    """

    try:
        response = requests.post(
            SPARQL_ENDPOINT,
            data={'query': query, 'format': 'application/sparql-results+json'},
            headers={
                'Accept': 'application/sparql-results+json',
                'User-Agent': 'Mozilla/5.0 (academic research)',
            },
            timeout=timeout,
        )

        if response.status_code != 200:
            return None, f'error_{response.status_code}'

        bindings = response.json()['results']['bindings']
        if not bindings:
            return None, 'not_found'

        return bindings[0]['title']['value'], 'ok'

    except requests.exceptions.Timeout:
        return None, 'timeout'
    except Exception:
        return None, 'error'


# Test
print("Test su 32019R0452...")
title, status = get_title_from_cellar('32019R0452')
print(f"  Status: {status}")
print(f"  Titolo: {title}")

## 3. Fetch con Checkpoint

Con 2.767 nodi e 0.5s di delay il tempo stimato e circa **25 minuti**. Se viene interrotto, riesegui questa cella: ripartira dal checkpoint automaticamente.

In [ ]:
results = []
errors  = []
total   = len(nodes_todo)

print(f"Inizio fetch: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    celex   = row['Label']
    node_id = row['Id']

    title, status = get_title_from_cellar(celex)

    results.append({
        'Id':           node_id,
        'Label':        celex,
        'title':        title,
        'title_status': status,
    })

    if status != 'ok':
        errors.append((celex, status))

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct      = (i + 1) / total * 100
        ok_count = sum(1 for r in results if r['title_status'] == 'ok')
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {ok_count}  errori: {len(errors)}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint_updated = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato ({len(checkpoint_updated)} nodi totali)")

    time.sleep(DELAY_SECONDS)

# Checkpoint finale
batch            = pd.DataFrame(results)
checkpoint_final = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)

print(f"\nFetch completato.")
print(f"  OK:        {(checkpoint_final['title_status'] == 'ok').sum()}")
print(f"  Not found: {(checkpoint_final['title_status'] == 'not_found').sum()}")
print(f"  Errori:    {(~checkpoint_final['title_status'].isin(['ok','not_found'])).sum()}")

## 4. Export

In [ ]:
titles_df    = pd.read_csv(checkpoint_file)[['Id', 'title', 'title_status']]
nodes_titled = nodes.merge(titles_df, on='Id', how='left')
nodes_titled.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"  Nodi totali:  {len(nodes_titled)}")
print(f"  Con titolo:   {nodes_titled['title'].notna().sum()} ({nodes_titled['title'].notna().sum()/len(nodes_titled)*100:.1f}%)")
print(f"  Senza titolo: {nodes_titled['title'].isna().sum()}")
print()
print("Esempi titoli per LegalType:")
for tipo in nodes_titled['LegalType'].dropna().unique():
    subset  = nodes_titled[(nodes_titled['LegalType'] == tipo) & nodes_titled['title'].notna()]
    esempio = subset['title'].iloc[0] if len(subset) > 0 else 'N/A'
    print(f"  {tipo:<20} {str(esempio)[:80]}")

## 5. Diagnostica

I nodi `not_found` sono tipicamente atti molto vecchi (anni '50-'60) o trattati con CELEX non standard. Per la classificazione nei layer verranno gestiti tramite fallback sul tipo di atto.

In [ ]:
print("Distribuzione status:")
print(nodes_titled['title_status'].value_counts().to_string())
print()

no_title = nodes_titled[nodes_titled['title'].isna()]
if len(no_title) > 0:
    print(f"Nodi senza titolo: {len(no_title)}")
    print("Per tipo:")
    print(no_title['LegalType'].value_counts().to_string())
    print()
    print("Per decade:")
    print(no_title['Decade'].value_counts().sort_index().to_string())
    print()
    print("Esempi CELEX senza titolo:")
    print(no_title['Label'].head(10).tolist())
else:
    print("Tutti i nodi hanno un titolo.")